In [29]:
pip install xgboost shap scikit-learn pandas matplotlib

1373.44s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [30]:
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [32]:
print("Loading synthetic dataset...")
df = pd.read_csv("data.csv")

Loading synthetic dataset...


# 1. Preparing the Data
We drop Company_Name as it's text, and the target variables that happen AFTER approval

In [33]:
X = df.drop(columns=['Company_Name', 'Loan_Approved', 'Approved_Limit_Cr', 'Interest_Rate_Pct'])
y = df['Loan_Approved']

 # Split into training and testing sets

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Training the XGBoost Model

In [35]:
print("Training XGBoost Classifier...")
model = xgb.XGBClassifier(
        n_estimators=100, 
        learning_rate=0.1, 
        max_depth=7, 
        random_state=42,
        eval_metric='logloss'
    )
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy on Test Set: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training XGBoost Classifier...

Model Accuracy on Test Set: 98.10%

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99       705
           1       0.94      1.00      0.97       295

    accuracy                           0.98      1000
   macro avg       0.97      0.99      0.98      1000
weighted avg       0.98      0.98      0.98      1000



# 3. Extract SHAP Values for Explainability

In [41]:
print("\nCalculating SHAP values (this is the 'Judge Walkthrough' engine)...")
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

# --- VISUALIZATION 1: The Global Summary Plot ---
 # This shows the judges what features matter most across ALL applications.
plt.figure(figsize=(30, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("CrediMitra: Global Feature Importance")
plt.tight_layout()
plt.savefig("shap_summary_global.png", dpi=300)
print("Saved global explainability plot to 'shap_summary_global.png'")
plt.clf()


Calculating SHAP values (this is the 'Judge Walkthrough' engine)...
Saved global explainability plot to 'shap_summary_global.png'


/var/folders/l9/5dqzm1yd4rj2k7h8nhb5r68w0000gn/T/ipykernel_64509/2393966619.py:8: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, X_test, show=False)


In [ ]:
# --- VISUALIZATION 2: The Local Waterfall Plot (Single Application) ---
# This explains exactly why ONE specific company was accepted or rejected.
# Let's pick the first company in the test set.
sample_index = 0
    
# Set up the plot
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[sample_index], show=False)
    
prediction_status = "Approved" if y_pred[sample_index] == 1 else "Rejected"
plt.title(f"CrediMitra Chain of Thought: Why this loan was {prediction_status}")
plt.tight_layout()
plt.savefig("shap_waterfall_single.png", dpi=300)
print(f"Saved specific application explainability plot to 'shap_waterfall_single.png'")

Saved specific application explainability plot to 'shap_waterfall_single.png'


In [ ]:
# 4. Save the model to plug into Streamlit later
model.save_model("model.json")

print("\nSuccess! Model saved as 'model.json'")


Success! Model saved as 'model.json'
